## 2. Interactive Packet Fuzzer

Inject candidate moves below and inspect the Layer 3 Host Firewall verdict:


In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path(".").resolve()))

from notebook_lib import GovernorAdapter

# Test 1: Clean Valid Packet
p1 = GovernorAdapter.validate_packet(
    contract="SYM-032",
    task="PROVE_CYCLE_0",
    payload={"op": "ASSERT", "claim": "x -> y", "evidence": "PIN_01"},
)
print(f"[Pinned] Packet 1 Admitted: {p1.admitted} (Hash: {p1.packet_hash[:16]}...)")

# Test 2: FW-MOVE Violation (Undeclared Verb)
p2 = GovernorAdapter.validate_packet(
    contract="SYM-032", task="PROVE_CYCLE_0", payload={"op": "SPECULATE", "claim": "x -> y"}
)
print(
    f"[Pinned] Packet 2 Admitted: {p2.admitted} | Refusal: {p2.refusal_code} | Reason: {p2.reason}"
)

# Test 3: FW-CARRIER Violation (Forbidden Unicode control character)
p3 = GovernorAdapter.validate_packet(
    contract="SYM-032", task="PROVE_CYCLE_0", payload={"op": "ASSERT", "claim": "x \x00 y"}
)
print(f"[Pinned] Packet 3 Admitted: {p3.admitted} | Refusal: {p3.refusal_code}")

In [ ]:
import ipywidgets as widgets
from IPython.display import clear_output

op_selector = widgets.Dropdown(
    options=[
        "ASSERT",
        "CUT",
        "REDUCE",
        "PROVE",
        "VERIFY",
        "RETRACT",
        "REFUSE",
        "LAND",
        "INVALID_OP",
    ],
    value="ASSERT",
    description="Operator:",
)
payload_text = widgets.Text(value="edge_1 -> 0", description="Payload:")
fuzz_btn = widgets.Button(description="Submit to Host", button_style="primary")
out_box = widgets.Output()


def on_fuzz_click(b):
    with out_box:
        clear_output()
        res = GovernorAdapter.validate_packet(
            contract="SYM-032",
            task="INTERACTIVE_TASK",
            payload={"op": op_selector.value, "content": payload_text.value},
        )
        if res.admitted:
            print(f"ADMITTED! Packet SHA-256 Digest: {res.packet_hash}")
        else:
            print(f"REFUSED! Code: {res.refusal_code}\nReason: {res.reason}")


fuzz_btn.on_click(on_fuzz_click)
widgets.VBox(
    [
        widgets.HTML("<h4>Interactive Packet Firewall Tester</h4>"),
        op_selector,
        payload_text,
        fuzz_btn,
        out_box,
    ]
)

## 3. Scope: CAN-Show vs. CANNOT-Show

> **CAN-Show**:
> - Demonstrates that the Three Firewalls operate fail-closed, intercepting non-whitelisted characters, undeclared operators, and broken hash links.
> - Demonstrates that valid packets produce deterministic SHA-256 digests.
>
> **CANNOT-Show**:
> - A pass on one packet is that packet's checks. It is not a proof of contract compliance.


## 4. Student Exercises

### Exercise 04.1: Fail-Closed Semantics
What does it mean for a firewall to operate "fail-closed"?

<details>
<summary><b>Click to reveal solution</b></summary>
A fail-closed firewall defaults to refusing admission whenever any check fails or whenever an unexpected input or exception occurs. Admission requires every required check to explicitly pass.
</details>


## 5. Anchors

- **Paper 032 Sections**: §4.1–§4.4
- **Ledger IDs**: `PO032-C22` through `PO032-C28`
- **Code**: `papers/032.../code/packet_firewall_host.py`


> **Claim-typing legend**
> - **[Pinned]**: taken from a file this repository already has.
> - **[Proved]**: a finite check in this notebook. Not a general proof.
> - **[Declared]**: a carrier, rule, or heading chosen for this sitting.
> - **[Open]**: something this sitting does not decide.
## Anchors

- Former notebook: `04_packet_firewall_host`
- The code checks one finite example. It is not a general proof.


In [ ]:
assert p1.admitted is True
assert p2.admitted is False
assert p3.admitted is False